[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-1/simple-graph.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58238187-lesson-2-simple-graph)

# 最简单的图

让我们构建一个包含3个节点和一个条件边的简单图。

![Screenshot 2024-08-20 at 3.11.22 PM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dba5f465f6e9a2482ad935_simple-graph1.png)

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langgraph

## 状态

首先，定义图的 [状态](https://langchain-ai.github.io/langgraph/concepts/low_level/#state)。

状态模式用作图中所有节点和边的输入模式。

让我们使用 Python 的 `typing` 模块中的 `TypedDict` 类作为我们的模式，它为键提供类型提示。

In [ ]:
from typing_extensions import TypedDict

class State(TypedDict):
    graph_state: str

## 节点

[节点](https://langchain-ai.github.io/langgraph/concepts/low_level/#nodes) 只是 Python 函数。

第一个位置参数是状态，如上面定义的。

因为状态是一个具有如上定义模式的 `TypedDict`，每个节点可以使用 `state['graph_state']` 访问键 `graph_state`。

每个节点返回状态键 `graph_state` 的新值。

默认情况下，每个节点返回的新值将 [覆盖](https://langchain-ai.github.io/langgraph/concepts/low_level/#reducers) 之前的状态值。

In [ ]:
def node_1(state):
    print("---节点 1---")
    return {"graph_state": state['graph_state'] +" 我是"}

def node_2(state):
    print("---节点 2---")
    return {"graph_state": state['graph_state'] +" 快乐的！"}

def node_3(state):
    print("---节点 3---")
    return {"graph_state": state['graph_state'] +" 悲伤的！"}

## 边

[边](https://langchain-ai.github.io/langgraph/concepts/low_level/#edges) 连接节点。

如果您想要 *总是* 从例如 `node_1` 转到 `node_2`，则使用普通边。

如果您想要在节点之间 *有选择地* 路由，则使用 [条件边](https://langchain-ai.github.io/langgraph/concepts/low_level/#conditional-edges)。

条件边作为函数实现，根据某些逻辑返回要访问的下一个节点。

In [ ]:
import random
from typing import Literal

def decide_mood(state) -> Literal["node_2", "node_3"]:
    
    # 通常，我们将使用状态来决定要访问的下一个节点
    user_input = state['graph_state'] 
    
    # 这里，让我们在节点 2、3 之间做一个 50/50 分割
    if random.random() < 0.5:

        # 50% 的时候，我们返回节点 2
        return "node_2"
    
    # 50% 的时候，我们返回节点 3
    return "node_3"

## 图构造

现在，我们从上面定义的 [组件](https://langchain-ai.github.io/langgraph/concepts/low_level/) 构建图。

[StateGraph 类](https://langchain-ai.github.io/langgraph/concepts/low_level/#stategraph) 是我们可以使用的图类。

首先，我们使用上面定义的 `State` 类初始化一个 StateGraph。

然后，我们添加我们的节点和边。

我们使用 [`START` 节点，一个特殊节点](https://langchain-ai.github.io/langgraph/concepts/low_level/#start-node)，它将用户输入发送到图，来指示我们的图从哪里开始。

[`END` 节点](https://langchain-ai.github.io/langgraph/concepts/low_level/#end-node) 是一个代表终端节点的特殊节点。

最后，我们 [编译我们的图](https://langchain-ai.github.io/langgraph/concepts/low_level/#compiling-your-graph) 来对图结构执行一些基本检查。

我们可以将图可视化为 [Mermaid 图表](https://github.com/mermaid-js/mermaid)。

In [ ]:
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END

# 构建图
builder = StateGraph(State)
builder.add_node("node_1", node_1)
builder.add_node("node_2", node_2)
builder.add_node("node_3", node_3)

# 逻辑
builder.add_edge(START, "node_1")
builder.add_conditional_edges("node_1", decide_mood)
builder.add_edge("node_2", END)
builder.add_edge("node_3", END)

# 添加
graph = builder.compile()

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

## 图调用

编译的图实现了 [runnable](https://python.langchain.com/docs/concepts/runnables/) 协议。

这提供了执行 LangChain 组件的标准方式。

`invoke` 是此接口中的标准方法之一。

输入是一个字典 `{"graph_state": "嗨，我是 Lance。"}`，它为我们的图状态字典设置初始值。

当调用 `invoke` 时，图从 `START` 节点开始执行。

它按顺序通过定义的节点（`node_1`、`node_2`、`node_3`）进行。

条件边将使用 50/50 决策规则从节点 `1` 遍历到节点 `2` 或 `3`。

每个节点函数接收当前状态并返回一个新值，这会覆盖图状态。

执行持续到达到 `END` 节点。

In [ ]:
graph.invoke({"graph_state" : "嗨，我是 Lance。"})

`invoke` 同步运行整个图。

这会等待每一步完成，然后再移动到下一步。

它在所有节点执行完毕后返回图的最终状态。

在这种情况下，它在 `node_3` 完成后返回状态：

```
{'graph_state': '嗨，我是 Lance。 我是 悲伤的！'}
```